In [4]:
import pandas as pd

df = pd.read_csv(r"Y:\Python\AI ML\Month 2\week 6-Classic ML\Machine Learning\supervised_learning\data\raw\bank-full.csv", sep =';')
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [5]:
df.shape

(45211, 17)

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   age        45211 non-null  int64
 1   job        45211 non-null  str  
 2   marital    45211 non-null  str  
 3   education  45211 non-null  str  
 4   default    45211 non-null  str  
 5   balance    45211 non-null  int64
 6   housing    45211 non-null  str  
 7   loan       45211 non-null  str  
 8   contact    45211 non-null  str  
 9   day        45211 non-null  int64
 10  month      45211 non-null  str  
 11  duration   45211 non-null  int64
 12  campaign   45211 non-null  int64
 13  pdays      45211 non-null  int64
 14  previous   45211 non-null  int64
 15  poutcome   45211 non-null  str  
 16  y          45211 non-null  str  
dtypes: int64(7), str(10)
memory usage: 5.9 MB


In [7]:
df['y'].value_counts()

y
no     39922
yes     5289
Name: count, dtype: int64

In [8]:
x = df.drop("y", axis=1)
y = df["y"]

In [9]:
x = x.drop("duration", axis=1)

In [10]:
categorical_columns = x.select_dtypes(include="object").columns
numerical_columns = x.select_dtypes(exclude="object").columns

print("Categorical:", list(categorical_columns))
print("Numerical:", list(numerical_columns))

Categorical: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']
Numerical: ['age', 'balance', 'day', 'campaign', 'pdays', 'previous']


C:\Users\yuvan\AppData\Local\Temp\ipykernel_22696\3270515302.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = x.select_dtypes(include="object").columns


In [12]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size= 0.2,
    random_state= 42,
    stratify= y
)

print(x_train.shape)
print(x_test.shape)

(36168, 15)
(9043, 15)


In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_columns)
    ],
    remainder="passthrough"
)

In [14]:
x_train_encoded = preprocessor.fit_transform(x_train)
x_test_encoded = preprocessor.transform(x_test)

print(x_train_encoded.shape)
print(x_test_encoded.shape)

(36168, 50)
(9043, 50)


In [15]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    random_state= 42
)

dt.fit(x_train_encoded,y_train)
dt_train_pred = dt.predict(x_train_encoded)
dt_test_pred = dt.predict(x_test_encoded)

In [16]:
from sklearn.metrics import accuracy_score, classification_report

print("Decision Tree")
print("Train Accuracy:", accuracy_score(y_train, dt_train_pred))
print("Test Accuracy :", accuracy_score(y_test, dt_test_pred))

print("\nClassification Report:")
print(classification_report(y_test, dt_test_pred))

Decision Tree
Train Accuracy: 1.0
Test Accuracy : 0.8329094327103838

Classification Report:
              precision    recall  f1-score   support

          no       0.91      0.90      0.90      7985
         yes       0.30      0.33      0.31      1058

    accuracy                           0.83      9043
   macro avg       0.61      0.61      0.61      9043
weighted avg       0.84      0.83      0.84      9043



In [17]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

rf.fit(x_train_encoded, y_train)

rf_train_pred = rf.predict(x_train_encoded)
rf_test_pred = rf.predict(x_test_encoded)

In [18]:
print("Random Forest")
print("Train Accuracy:", accuracy_score(y_train, rf_train_pred))
print("Test Accuracy :", accuracy_score(y_test, rf_test_pred))

print("\nClassification Report:")
print(classification_report(y_test, rf_test_pred))

Random Forest
Train Accuracy: 1.0
Test Accuracy : 0.8938405396439235

Classification Report:
              precision    recall  f1-score   support

          no       0.91      0.98      0.94      7985
         yes       0.62      0.24      0.34      1058

    accuracy                           0.89      9043
   macro avg       0.76      0.61      0.64      9043
weighted avg       0.87      0.89      0.87      9043



In [20]:
from sklearn.metrics import confusion_matrix
print("Decision Tree:")
print(confusion_matrix(y_test, dt_test_pred))

print("\nRandom Forest:")
print(confusion_matrix(y_test, rf_test_pred))

Decision Tree:
[[7188  797]
 [ 714  344]]

Random Forest:
[[7832  153]
 [ 807  251]]


In [21]:
print("Decision Tree balanced accuracy:",
      __import__("sklearn").metrics.balanced_accuracy_score(y_test, dt_test_pred))

print("Random Forest balanced accuracy:",
      __import__("sklearn").metrics.balanced_accuracy_score(y_test, rf_test_pred))

Decision Tree balanced accuracy: 0.612664814580268
Random Forest balanced accuracy: 0.6090395744383669


In [22]:
rf_oob = RandomForestClassifier(
    n_estimators=200,
    max_features="sqrt",
    bootstrap=True,
    oob_score=True,
    random_state=42,
    n_jobs=-1
)

rf_oob.fit(x_train_encoded, y_train)

print("Random Forest OOB Score:", rf_oob.oob_score_)

Random Forest OOB Score: 0.8933034726830347


In [24]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

depths = [1, 2, 3, 5, 10, 20, None]

for depth in depths:
    model = DecisionTreeClassifier(
        max_depth=depth,
        random_state=42
    )

    model.fit(x_train_encoded, y_train)

    train_pred = model.predict(x_train_encoded)
    test_pred = model.predict(x_test_encoded)

    train_acc = accuracy_score(y_train, train_pred)
    test_acc = accuracy_score(y_test, test_pred)

    print(
        f"Depth={depth} | "
        f"Train={train_acc:.4f} | "
        f"Test={test_acc:.4f}"
    )

Depth=1 | Train=0.8928 | Test=0.8932
Depth=2 | Train=0.8928 | Test=0.8932
Depth=3 | Train=0.8937 | Test=0.8932
Depth=5 | Train=0.8946 | Test=0.8927
Depth=10 | Train=0.9080 | Test=0.8915
Depth=20 | Train=0.9477 | Test=0.8767
Depth=None | Train=1.0000 | Test=0.8329


# Random Forest — Complete Notes

## 1. What is Random Forest?

**Random Forest** is an ensemble learning algorithm that combines many Decision Trees to produce a more stable and usually better-generalizing model.

Instead of relying on one tree:

```text
                Training Data
                     ↓
        ┌────────────┼────────────┐
        ↓            ↓            ↓
      Tree 1       Tree 2       Tree 3   ... many trees
        ↓            ↓            ↓
        └────────────┼────────────┘
                     ↓
               Aggregation
                     ↓
                 Prediction
```

Random Forest =

> **Decision Trees + Bootstrap Sampling + Random Feature Selection + Aggregation**

---

# 2. Why Random Forest?

A single Decision Tree can have **high variance**.

Small changes in training data can produce a substantially different tree.

It may behave like:

```text
Train accuracy → 100%
Test accuracy  → much lower
```

This is overfitting.

Random Forest addresses this by building **many different trees** and combining their predictions.

### Main idea

> Individual trees can be noisy, but averaging/voting many diverse trees can produce a more stable prediction.

---

# 3. Ensemble Learning

An **ensemble** combines multiple models to create a stronger overall model.

Two major approaches:

### Bagging

Models are trained **independently** on different samples of the training data.

Example:

```text
Data
 ↓
Bootstrap 1 → Tree 1
Bootstrap 2 → Tree 2
Bootstrap 3 → Tree 3
...
```

Random Forest uses **bagging + random feature selection**.

### Boosting

Models are trained **sequentially**, with later models focusing more on previous errors.

Examples include:

* AdaBoost
* Gradient Boosting
* XGBoost

Don't confuse:

> **Bagging = parallel/independent models**

> **Boosting = sequential/error-focused models**

---

# 4. Bootstrap Sampling

Bootstrap sampling means:

> **Random sampling with replacement**

Suppose the original dataset is:

```text
A B C D E
```

A bootstrap sample could be:

```text
A C C E A
```

Notice:

* Same number of samples as original
* Duplicates are allowed
* Some original samples may not appear

Each tree gets its **own bootstrap sample**.

---

# 5. Out-of-Bag Samples

Samples that weren't selected for a particular tree are called its **Out-of-Bag (OOB)** samples.

Example:

```text
Original:
A B C D E

Tree's bootstrap sample:
A C C E A

OOB:
B D
```

These OOB samples weren't used to train that particular tree.

Therefore, they can be used to estimate how that tree/forest performs on unseen training examples.

---

# 6. Random Feature Selection

Random Forest adds another source of randomness.

At **each split**, the tree randomly selects a subset of features and searches for the best split only among those features.

Suppose:

```text
20 features
max_features = 5
```

At one node:

```text
Randomly selected:
F2 F7 F11 F15 F19
```

The best split is chosen from those 5.

At another node, a different subset can be selected.

### Important

`max_features=5` does **not** mean:

> The entire tree can only use 5 features.

It means:

> **5 candidate features are considered at each split.**

---

# 7. Why Random Feature Selection?

Imagine every tree always chooses the same strongest feature.

Then the trees could become very similar.

Similar trees make similar mistakes.

Random feature selection encourages **diversity** among trees.

```text
Bootstrap randomness
        +
Feature randomness
        ↓
Different trees
        ↓
Less correlated errors
        ↓
Better ensemble
```

This is a key reason Random Forest works well.

---

# 8. How Random Forest Works

The complete algorithm:

### Step 1 — Create bootstrap samples

For every tree:

```text
Training data
     ↓
Bootstrap sample
```

### Step 2 — Build a Decision Tree

At every node:

```text
Randomly select feature subset
          ↓
Find best split among them
          ↓
Split
```

### Step 3 — Repeat

Build many trees:

```text
Tree 1
Tree 2
Tree 3
...
Tree 200
```

### Step 4 — Aggregate predictions

Classification:

> **Majority voting**

Regression:

> **Average**

---

# 9. Classification — Majority Voting

Suppose five trees predict:

```text
Tree 1 → Yes
Tree 2 → No
Tree 3 → Yes
Tree 4 → Yes
Tree 5 → No
```

Votes:

```text
Yes = 3
No  = 2
```

Final prediction:

```text
Yes
```

---

# 10. Regression — Averaging

Suppose five trees predict:

```text
20
24
22
26
28
```

Random Forest prediction:

$$
\frac{20+24+22+26+28}{5}=24
$$

So:

```text
Classification → majority vote
Regression     → average
```

---

# 11. Why Random Forest Reduces Variance

A Decision Tree can be highly sensitive to training data.

Random Forest:

```text
Many trees
    ↓
Different bootstrap samples
    +
Different feature subsets
    ↓
Diverse trees
    ↓
Aggregate predictions
    ↓
Errors partially cancel
    ↓
Lower variance
    ↓
Better generalization
```

### Important distinction

Random Forest doesn't necessarily make **each individual tree** less overfit.

Instead:

> **The ensemble becomes more stable because many diverse trees are aggregated.**

---

# 12. Bias–Variance Intuition

Generally:

| Model            |                                           Bias | Variance |
| ---------------- | ---------------------------------------------: | -------: |
| Very simple tree |                                           High |      Low |
| Deep tree        |                                            Low |     High |
| Random Forest    | Usually lower variance than a single deep tree |    Lower |

The goal isn't:

> "Get the highest training accuracy."

The goal is:

> **Good performance on unseen data.**

---

# 13. Out-of-Bag (OOB) Evaluation

Enable OOB evaluation:

```python id="9p0w3j"
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    bootstrap=True,
    oob_score=True,
    random_state=42
)

model.fit(x_train, y_train)

print(model.oob_score_)
```

### What happens?

Each training sample is OOB for some subset of trees.

Those trees make predictions for that sample.

These predictions can be aggregated to produce an OOB estimate.

### Important

OOB evaluation:

* uses the training data internally
* is useful for estimating generalization
* doesn't require a separate validation split for that purpose

But:

> **OOB does NOT replace the final untouched test set.**

---

# 14. `n_estimators`

Number of trees in the forest.

```python id="cbt6fy"
RandomForestClassifier(
    n_estimators=200
)
```

means:

> **200 trees**

Generally:

```text
n_estimators ↑
       ↓
More stable predictions
       ↓
Variance tends to decrease
       ↓
Computation ↑
```

Eventually there are **diminishing returns**.

Increasing from:

```text
10 → 100
```

may help considerably.

Increasing:

```text
500 → 1000
```

may provide very little improvement depending on the problem.

---

# 15. `max_features`

Controls how many features are considered at each split.

Common options:

```python id="t2p8ju"
max_features="sqrt"
max_features="log2"
max_features=None
max_features=5
max_features=0.5
```

For classification, `"sqrt"` is a common default choice.

### Effect

Smaller `max_features`:

* More randomness
* More tree diversity
* Potentially lower correlation
* Can increase bias if too restrictive

Larger `max_features`:

* Trees become more similar
* Less randomness
* Potentially lower bias
* Potentially higher correlation

---

# 16. `max_depth`

Controls the maximum depth of **each individual tree**.

```python id="5hsv83"
RandomForestClassifier(max_depth=10)
```

Smaller depth:

```text
simpler trees
bias ↑
variance ↓
```

Larger depth:

```text
more complex trees
bias ↓
variance ↑
```

Random Forest can still use deep trees because aggregation helps control the overall variance.

But **unlimited complexity isn't automatically optimal**.

---

# 17. `min_samples_split`

Minimum number of samples required in a node before it can be split.

```python id="w1fmxn"
min_samples_split=10
```

A node needs at least 10 samples before a split can be considered.

Increasing it generally:

```text
complexity ↓
variance ↓
bias ↑
```

### Don't confuse with `min_samples_leaf`

`min_samples_split`:

> Can this current node split?

`min_samples_leaf`:

> Are the resulting leaves allowed to be this small?

---

# 18. `min_samples_leaf`

Minimum number of samples that must exist in each resulting leaf.

```python id="0y3pmw"
min_samples_leaf=5
```

A candidate split producing:

```text
12 samples | 3 samples
```

would be rejected.

A split producing:

```text
12 samples | 8 samples
```

would be allowed.

Increasing it generally reduces tree complexity and variance.

---

# 19. `bootstrap`

Controls whether bootstrap samples are used.

```python id="4w7g6c"
bootstrap=True
```

means:

> Each tree receives a bootstrap sample.

```python
bootstrap=False
```

removes that bootstrap row-sampling mechanism.

Also:

> **OOB evaluation requires bootstrap sampling.**

---

# 20. `class_weight`

Useful for imbalanced classification.

Example:

```text
Class A = 900
Class B = 100
```

Using:

```python
class_weight="balanced"
```

gives greater importance to the minority class during training.

### Important

It does **not** create additional samples.

It changes the importance/cost associated with classes.

---

# 21. Feature Scaling

Unlike:

* KNN
* SVM
* Logistic Regression
* Ridge

Random Forest generally **doesn't require feature scaling**.

Why?

Trees make decisions using conditions such as:

```text
age <= 35
balance <= 1500
```

They don't depend on Euclidean distance or coefficient magnitude.

So:

```text
StandardScaler → not required for Random Forest
```

---

# 22. Feature Importance

Random Forest provides:

```python id="j8eg4d"
model.feature_importances_
```

This gives **impurity-based feature importance**.

It represents how much features contributed to reducing impurity across the forest.

### Important ⚠️

Feature importance does **not** mean:

> "This feature causes the target."

And it doesn't necessarily mean:

> "This feature is independently the most important in the real world."

---

# 23. Permutation Importance

Another approach is **permutation importance**.

Concept:

```text
Original test data
       ↓
Measure performance
       ↓
Shuffle one feature
       ↓
Measure performance again
       ↓
Performance drop = importance
```

If shuffling a feature causes a large performance decrease:

> The model relied heavily on that feature.

⚠️ Correlated features can make permutation importance harder to interpret because another correlated feature may contain similar information.

---

# 24. Decision Tree vs Random Forest

|                          | Decision Tree | Random Forest   |
| ------------------------ | ------------- | --------------- |
| Number of trees          | One           | Many            |
| Bootstrap                | ❌             | ✅ usually       |
| Random feature selection | ❌             | ✅               |
| Variance                 | High          | Generally lower |
| Overfitting risk         | Higher        | Generally lower |
| Interpretability         | High          | Lower           |
| Computation              | Lower         | Higher          |
| Prediction stability     | Lower         | Higher          |

Random Forest isn't **automatically** better in every dataset.

Use validation/CV and appropriate metrics.

---

# 25. Our Real Dataset Experiment

We used the **Bank Marketing dataset**:

```text
45,211 rows
17 columns
16 original features + target
```

We removed:

```text
duration
```

because it represents the duration of the current call and can introduce unrealistic information if the model is supposed to predict subscription before the call's final duration is known.

After one-hot encoding:

```text
36,168 × 50 → training
 9,043 × 50 → testing
```

The target was highly imbalanced:

```text
no  → 39,922
yes →  5,289
```

Approximately:

```text
no  → 88.3%
yes → 11.7%
```

---

# 26. Decision Tree vs Random Forest Results

### 🌳 Decision Tree

```text
Train Accuracy = 1.000
Test Accuracy  = 0.833
```

This large gap demonstrated substantial overfitting.

For the minority `yes` class:

```text
Precision = 0.30
Recall    = 0.33
F1        = 0.31
```

### 🌲 Random Forest

```text
Train Accuracy = 1.000
Test Accuracy  = 0.894
```

So test accuracy improved:

```text
83.3% → 89.4%
```

The Random Forest generalized better on this particular split.

However:

```text
yes precision = 0.62
yes recall    = 0.24
yes F1        = 0.34
```

So it became much more precise when predicting `yes`, but it caught fewer actual `yes` customers.

---

# 27. Confusion Matrix Interpretation

### Decision Tree

```text
[[7188  797]
 [ 714  344]]
```

For `yes`:

```text
TP = 344
FN = 714
```

### Random Forest

```text
[[7832  153]
 [ 807  251]]
```

For `yes`:

```text
TP = 251
FN = 807
```

Random Forest dramatically reduced false positives:

```text
797 → 153
```

but increased false negatives:

```text
714 → 807
```

Therefore, the "better" model depends on the business objective.

---

# 28. Why Accuracy Was Misleading

Our dataset was approximately:

```text
88.3% no
11.7% yes
```

A model that predicts `no` for almost everybody can already achieve high accuracy.

Therefore, for this problem we should inspect:

* Precision
* Recall
* F1
* Confusion matrix
* Balanced accuracy
* ROC-AUC / PR-AUC when appropriate

rather than relying on accuracy alone.

Our balanced accuracy was approximately:

```text
Decision Tree → 0.613
Random Forest → 0.609
```

So although RF had substantially higher overall accuracy, its balanced accuracy was slightly lower on this particular split.

**This is an excellent example of why metric selection depends on the problem.**

---

# 29. OOB Result

Our Random Forest produced:

```text
OOB Score = 0.8933
Test Accuracy = 0.8938
```

They were extremely close in this experiment.

That demonstrates how OOB can provide a useful internal estimate of generalization.

But don't memorize:

> "OOB should always equal test accuracy."

It won't necessarily.

---

# 30. Depth Experiment

We tested a single Decision Tree:

| Depth |     Train |      Test |
| ----: | --------: | --------: |
|     1 |     0.893 |     0.893 |
|     2 |     0.893 |     0.893 |
|     3 |     0.894 |     0.893 |
|     5 |     0.895 |     0.893 |
|    10 |     0.908 |     0.892 |
|    20 |     0.948 |     0.877 |
|  None | **1.000** | **0.833** |

This showed:

```text
Tree complexity ↑
       ↓
Training performance ↑
       ↓
Eventually test performance ↓
       ↓
Overfitting
```

The unrestricted tree reached:

```text
100% train
83.3% test
```

🔥 This was our practical demonstration of **high variance**.

---

# 31. Important Mental Model

Remember Random Forest as three things:

```text
          RANDOM FOREST
                │
     ┌──────────┼──────────┐
     ↓          ↓          ↓
Bootstrap   Random      Aggregation
  Rows      Features
     │          │          │
     └──────────┼──────────┘
                ↓
        Diverse Trees
                ↓
       Stable Prediction
                ↓
       Lower Variance
```

### The one-line version:

> **Random Forest builds many diverse Decision Trees using bootstrap samples and random feature subsets, then combines their predictions to reduce variance and improve generalization.**

---

# 32. Common Mistakes ⚠️

### ❌ "Random Forest chooses the best tree."

No.

It combines predictions from **many trees**.

### ❌ "`max_features=5` means the whole forest only uses 5 features."

No.

Five candidate features are considered **at each split**.

### ❌ "Random Forest prevents every tree from overfitting."

Not necessarily.

Individual trees can still be very complex.

### ❌ "More trees always makes the model better."

No.

Performance usually stabilizes, with diminishing returns.

### ❌ "OOB replaces the test set."

No.

Keep a final untouched test set.

### ❌ "Random Forest requires StandardScaler."

Generally no.

### ❌ "Feature importance means causation."

No.

It measures model reliance/contribution, not causality.

### ❌ "89% accuracy means 89% of predictions are equally good."

Not necessarily, especially with class imbalance.

---

# 33. Industry Workflow

A sensible Random Forest workflow:

```text
Raw Data
   ↓
Data validation
   ↓
Train/Test Split
   ↓
Preprocessing
   ↓
Pipeline
   ↓
Baseline Decision Tree
   ↓
Random Forest
   ↓
Cross-validation
   ↓
Hyperparameter tuning
   ↓
Evaluate appropriate metrics
   ↓
Error analysis
   ↓
Feature importance / interpretation
   ↓
Final untouched test evaluation
```

For classification with imbalance:

```text
Accuracy
+ Precision
+ Recall
+ F1
+ Confusion Matrix
+ Balanced Accuracy
+ ROC-AUC / PR-AUC
```

should be considered according to the problem.

---

# 34. Interview Questions

### What is Random Forest?

> Random Forest is an ensemble of Decision Trees that uses bootstrap sampling and random feature selection to create diverse trees and aggregates their predictions to reduce variance and improve generalization.

### Why does Random Forest usually outperform a single Decision Tree?

> A single tree can have high variance. Random Forest averages or votes across many diverse trees, reducing the effect of individual tree errors.

### Why random features?

> To reduce correlation between trees and increase ensemble diversity.

### What is bootstrap sampling?

> Random sampling with replacement from the training data.

### What is OOB evaluation?

> An internal validation estimate using samples that were not included in the bootstrap sample for particular trees.

### Does Random Forest need feature scaling?

> Generally no, because tree splits aren't based on distance or coefficient magnitude.

### What does `n_estimators` control?

> The number of trees in the forest.

### Difference between `max_depth` and `n_estimators`?

> `max_depth` controls the complexity of individual trees; `n_estimators` controls how many trees are combined.

### What happens when `max_features` decreases?

> More randomness and potentially less correlation between trees, but excessive restriction can increase bias.

---

# 🧠 Final Cheat Sheet

```text
Decision Tree
     ↓
High variance
     ↓
Random Forest
     ↓
Many trees
     +
Bootstrap rows
     +
Random feature subsets
     ↓
Diverse trees
     ↓
Aggregation
     ↓
Lower variance
     ↓
Better generalization
```

### Core parameters

```text
n_estimators       → number of trees
max_features       → candidate features per split
max_depth          → tree depth
min_samples_split  → samples needed to split a node
min_samples_leaf   → minimum samples in a leaf
bootstrap           → use bootstrap samples?
criterion            → split quality
class_weight        → handle class imbalance
```

### Core distinction

```text
Decision Tree → one tree
Random Forest → many diverse trees + aggregation
```
